# Phase 6A Screening Matrix

Parameterized screening notebook for Modal execution. Inputs come from environment variables.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os

from phase6_screening_common import (
    assert_npu_compatible,
    build_phase6_data_bundle,
    configure_runtime,
    choose_amp_policy,
    env_flag,
    env_int,
    export_to_onnx,
    fit_stage,
    get_model_spec,
    print_data_summary,
    resolve_phase6_workspace,
    run_diagnostics,
)

MODEL_ID = os.environ.get("LAB2_MODEL_ID", "wide_se")
PRETRAIN_MIX = os.environ.get("LAB2_PRETRAIN_MIX", "coco_only")
STAGE_NAME = os.environ.get("LAB2_STAGE", "stage1_pretrain")
EPOCHS = env_int("LAB2_STAGE_EPOCHS", 12 if STAGE_NAME == "stage1_pretrain" else 8)
BATCH_SIZE = env_int("LAB2_BATCH_SIZE", 4)
NUM_WORKERS = env_int("LAB2_NUM_WORKERS", 2)
SEED = env_int("LAB2_SEED", 255)
RESUME_TRAINING = env_flag("LAB2_RESUME_TRAINING", True)
RUN_DIAGNOSTICS = env_flag("LAB2_RUN_DIAGNOSTICS", True)
RUN_ONNX_EXPORT = env_flag("LAB2_RUN_ONNX_EXPORT", False)
USE_AMP = env_flag("LAB2_USE_AMP", True)
CHANNELS_LAST = env_flag("LAB2_CHANNELS_LAST", True)
INIT_CHECKPOINT = os.environ.get("LAB2_INIT_CHECKPOINT")

workspace = resolve_phase6_workspace(output_subdir="phase6_screening")
device = configure_runtime()
amp_policy = choose_amp_policy(device)
if not USE_AMP:
    amp_policy = {"enabled": False, "dtype": None, "use_scaler": False, "label": "fp32"}
channels_last = bool(CHANNELS_LAST and device.type == "cuda")

spec = get_model_spec(MODEL_ID)
stage_cfg = {
    "stage_name": STAGE_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": 3e-4 if STAGE_NAME == "stage1_pretrain" else 1.5e-4,
    "weight_decay": 2e-4,
    "warmup_epochs": 2 if STAGE_NAME == "stage1_pretrain" else 1,
    "min_lr_ratio": 0.05,
    "checkpoint_interval": 4 if STAGE_NAME == "stage1_pretrain" else 2,
    "train_eval_interval": 2 if STAGE_NAME == "stage1_pretrain" else 1,
    "seed": SEED,
    "early_stop_patience": 6 if STAGE_NAME == "stage1_pretrain" else 5,
    "grad_clip_norm": 1.0,
    "ema_decay": 0.999,
    "charb_eps": 1e-6,
    "selection_metric": "paired_val_psnr",
}
data_cfg = {
    "train_patch_size": 224,
    "eval_size": 256,
    "random_scale_pad": 48,
    "cutout_prob": 0.35,
    "cutout_ratio": 0.18,
    "lr_noise_prob": 0.30,
    "lr_noise_std": 0.02,
    "lr_blur_prob": 0.80,
    "blur_radius_min": 0.2,
    "blur_radius_max": 1.6,
    "jpeg_prob": 0.60,
    "jpeg_quality_min": 25,
    "jpeg_quality_max": 90,
    "downsample_scales": (2, 3, 4),
    "resize_modes": ("bicubic", "bilinear", "lanczos"),
    "imagenet_train_limit": 6000,
    "imagenet_val_limit": 300,
    "coco_train_limit": 12000,
    "coco_val_limit": 500,
    "train_eval_subset_size": 128,
}

print(json.dumps({
    "model_id": MODEL_ID,
    "mix_id": PRETRAIN_MIX,
    "stage_name": STAGE_NAME,
    "stage_cfg": stage_cfg,
    "data_cfg": data_cfg,
    "workspace": {k: str(v) for k, v in workspace.items()},
    "params": spec["params"],
    "ops": spec["ops"],
}, indent=2))

model_for_check = spec["build_model"](**spec["model_cfg"])
assert_npu_compatible(model_for_check)

data_bundle = build_phase6_data_bundle(
    workspace=workspace,
    data_cfg=data_cfg,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    device=device,
    seed=SEED,
    pretrain_mix=PRETRAIN_MIX,
    stage_name=STAGE_NAME,
)
print_data_summary(data_bundle)

history = fit_stage(
    model=spec["build_model"](**spec["model_cfg"]),
    train_loader=data_bundle["train_loader"],
    train_eval_loader=data_bundle["train_eval_loader"],
    eval_loaders={
        "paired_val": data_bundle["paired_val_loader"],
        "combined_val": data_bundle["combined_val_loader"],
        "coco_val": data_bundle["coco_val_loader"],
        "imagenet_val": data_bundle["imagenet_val_loader"],
    },
    output_dir=Path(workspace["output_dir"]),
    model_cfg=spec["model_cfg"],
    train_cfg=stage_cfg,
    data_cfg=data_cfg,
    device=device,
    amp_policy=amp_policy,
    model_id=MODEL_ID,
    mix_id=PRETRAIN_MIX,
    stage_name=STAGE_NAME,
    channels_last=channels_last,
    resume=RESUME_TRAINING,
    init_checkpoint_path=Path(INIT_CHECKPOINT) if INIT_CHECKPOINT else None,
)
print(f"Recorded epochs: {len(history)}")

if RUN_DIAGNOSTICS:
    run_diagnostics(
        build_model=spec["build_model"],
        model_cfg=spec["model_cfg"],
        output_dir=Path(workspace["output_dir"]),
        data_bundle=data_bundle,
        device=device,
        prepare_export_model=spec["prepare_export_model"],
    )

if RUN_ONNX_EXPORT:
    export_to_onnx(
        build_model=spec["build_model"],
        model_cfg=spec["model_cfg"],
        checkpoint_path=Path(workspace["output_dir"]) / "best.pt",
        onnx_path=Path(workspace["output_dir"]) / "best.onnx",
        data_cfg=data_cfg,
        device=device,
        prepare_export_model=spec["prepare_export_model"],
        verify=True,
        sample_loader=data_bundle["paired_val_loader"],
    )
